# Module 0.1: Markov Chain Basics

## Learning Objectives
By completing this notebook, you will:
1. Understand the Markov property and its implications for time series modeling
2. Construct and analyze transition matrices for discrete state spaces
3. Compute multi-step transition probabilities using matrix powers
4. Calculate stationary distributions and interpret their meaning
5. Simulate Markov chain trajectories for financial regime modeling

## Prerequisites
- Linear algebra (matrix multiplication, eigenvalues)
- Probability theory (conditional probability, expected values)
- Python programming basics

## Estimated Time: 45 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Linear algebra (matrix multiplication, eigenvalues)",
    "Probability theory (conditional probability, expected values)",
    "Python programming basics"
])

## Setup and Imports

We'll use NumPy for matrix operations, Matplotlib for visualization, and set random seeds for reproducibility.

In [ ]:
# Core numerical and visualization libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. The Markov Property

A stochastic process has the **Markov property** if the future state depends only on the current state, not on the history of states.

### Mathematical Definition

$$P(S_{t+1} = s_{t+1} | S_t = s_t, S_{t-1} = s_{t-1}, ..., S_0 = s_0) = P(S_{t+1} = s_{t+1} | S_t = s_t)$$

This is a **memoryless** property - the past is irrelevant given the present.

### Why This Matters in Finance

While real markets have memory (autocorrelation, momentum), Markov models provide:
- Tractable mathematical framework
- Foundation for regime-switching models
- Computational efficiency
- Interpretable state dynamics

## 2. Transition Matrices

The transition matrix $A$ encodes the probabilities of moving between states.

For a Markov chain with $K$ states, $A$ is $K \times K$ where:
$$a_{ij} = P(S_{t+1} = j | S_t = i)$$

### Properties
1. **Non-negative**: $a_{ij} \geq 0$ for all $i, j$
2. **Row-stochastic**: $\sum_j a_{ij} = 1$ for all $i$

In [ ]:
# Purpose: Create a simple market regime transition matrix
# Key Concept: Bull and Bear markets persist with high probability

def create_market_regime_transition_matrix():
    """
    Create a 2-state transition matrix for Bull and Bear regimes.
    
    States:
    - 0: Bull market (positive expected returns)
    - 1: Bear market (negative expected returns)
    
    Returns
    -------
    A : ndarray (2, 2)
        Transition matrix where A[i,j] = P(state_t+1 = j | state_t = i)
    """
    # Bull state tends to persist (90% probability)
    # Bull -> Bear transition is rare (10% probability)
    p_bull_to_bull = 0.90
    p_bull_to_bear = 0.10
    
    # Bear state is more volatile (70% persistence)
    # Bear -> Bull recovery is more likely (30% probability)
    p_bear_to_bear = 0.70
    p_bear_to_bull = 0.30
    
    A = np.array([
        [p_bull_to_bull, p_bull_to_bear],  # From Bull state
        [p_bear_to_bull, p_bear_to_bear]   # From Bear state
    ])
    
    return A

# Create and verify the transition matrix
A = create_market_regime_transition_matrix()

print("Market Regime Transition Matrix:")
print("="*50)
print("\nState 0 = Bull, State 1 = Bear\n")
print(A)
print("\nRow sums (should be 1.0):")
print(A.sum(axis=1))

# Visualize as heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(A, annot=True, fmt='.2f', cmap='Blues', 
            xticklabels=['Bull', 'Bear'],
            yticklabels=['Bull', 'Bear'],
            cbar_kws={'label': 'Transition Probability'},
            ax=ax)
ax.set_xlabel('To State', fontsize=12)
ax.set_ylabel('From State', fontsize=12)
ax.set_title('Market Regime Transition Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Multi-Step Transitions

To compute the probability of being in state $j$ after $n$ steps starting from state $i$, we use the **Chapman-Kolmogorov equation**:

$$P(S_{t+n} = j | S_t = i) = [A^n]_{ij}$$

This is a consequence of the Markov property and allows us to forecast future state probabilities.

In [ ]:
# Purpose: Analyze how regime probabilities evolve over time
# Key Concept: Matrix powers give multi-step transition probabilities

def compute_multistep_transitions(A: np.ndarray, max_steps: int = 20) -> List[np.ndarray]:
    """
    Compute transition probabilities over multiple time steps.
    
    Parameters
    ----------
    A : ndarray (K, K)
        Transition matrix
    max_steps : int
        Maximum number of steps to compute
    
    Returns
    -------
    powers : list of ndarray
        List where powers[n] = A^n
    """
    K = A.shape[0]
    powers = [np.eye(K)]  # A^0 = Identity matrix
    
    # Step 1: Compute each power by multiplying previous power by A
    for step in range(1, max_steps + 1):
        powers.append(powers[-1] @ A)
    
    return powers

# Compute multi-step transitions
max_steps = 20
powers = compute_multistep_transitions(A, max_steps)

# Extract probabilities starting from Bull state
prob_bull = [powers[n][0, 0] for n in range(max_steps + 1)]  # P(Bull at time n | Bull at time 0)
prob_bear = [powers[n][0, 1] for n in range(max_steps + 1)]  # P(Bear at time n | Bull at time 0)

# Visualize convergence
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(max_steps + 1), prob_bull, 'o-', label='P(Bull | start in Bull)', linewidth=2)
ax.plot(range(max_steps + 1), prob_bear, 's-', label='P(Bear | start in Bull)', linewidth=2)
ax.axhline(0.75, color='red', linestyle='--', alpha=0.5, label='Long-run Bull prob')
ax.axhline(0.25, color='orange', linestyle='--', alpha=0.5, label='Long-run Bear prob')
ax.set_xlabel('Time Steps', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Regime Probabilities Over Time (Starting in Bull State)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAfter {max_steps} steps:")
print(f"  P(Bull | started in Bull) = {prob_bull[-1]:.4f}")
print(f"  P(Bear | started in Bull) = {prob_bear[-1]:.4f}")

## 4. Stationary Distribution

A **stationary distribution** $\pi$ satisfies:
$$\pi = \pi A$$

This represents the long-run proportion of time spent in each state, independent of the starting state.

### Finding the Stationary Distribution

$\pi$ is a **left eigenvector** of $A$ with eigenvalue 1:
$$\pi^T A^T = \pi^T$$

In [ ]:
# Purpose: Compute the long-run regime probabilities
# Key Concept: Stationary distribution gives equilibrium probabilities

def compute_stationary_distribution(A: np.ndarray) -> np.ndarray:
    """
    Compute stationary distribution using eigenvalue decomposition.
    
    Parameters
    ----------
    A : ndarray (K, K)
        Transition matrix
    
    Returns
    -------
    pi : ndarray (K,)
        Stationary distribution
    """
    # Step 1: Compute eigenvalues and eigenvectors of A^T
    eigenvalues, eigenvectors = np.linalg.eig(A.T)
    
    # Step 2: Find the eigenvector corresponding to eigenvalue = 1
    idx = np.argmin(np.abs(eigenvalues - 1.0))
    pi = np.real(eigenvectors[:, idx])
    
    # Step 3: Normalize to sum to 1 (probability distribution)
    pi = pi / pi.sum()
    
    return pi

# Compute stationary distribution
pi = compute_stationary_distribution(A)

print("Stationary Distribution:")
print("="*50)
print(f"  π(Bull) = {pi[0]:.4f}")
print(f"  π(Bear) = {pi[1]:.4f}")
print(f"\nSum: {pi.sum():.4f}")

# Verify: π = πA
pi_check = pi @ A
print("\nVerification (π should equal πA):")
print(f"  π     = {pi}")
print(f"  πA    = {pi_check}")
print(f"  Error = {np.max(np.abs(pi - pi_check)):.2e}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['Bull', 'Bear'], pi, color=['green', 'red'], alpha=0.7, edgecolor='black')
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Stationary Distribution of Market Regimes', fontsize=14)
ax.set_ylim([0, 1])
for i, v in enumerate(pi):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Simulating Markov Chains

To build intuition and test implementations, we simulate trajectories from the Markov chain.

**Algorithm:**
1. Start in an initial state
2. At each time step, sample the next state from the transition probabilities
3. Repeat for the desired number of steps

In [ ]:
# Purpose: Simulate market regime trajectories
# Key Concept: Sample from transition probabilities to generate paths

def simulate_markov_chain(
    A: np.ndarray, 
    initial_state: int, 
    n_steps: int,
    random_seed: int = None
) -> np.ndarray:
    """
    Simulate a trajectory from a Markov chain.
    
    Parameters
    ----------
    A : ndarray (K, K)
        Transition matrix
    initial_state : int
        Starting state (0 to K-1)
    n_steps : int
        Number of steps to simulate
    random_seed : int, optional
        Random seed for reproducibility
    
    Returns
    -------
    states : ndarray (n_steps,)
        Simulated state sequence
    """
    if random_seed is not None:
        np.random.seed(random_seed)
    
    K = A.shape[0]
    states = np.zeros(n_steps, dtype=int)
    states[0] = initial_state
    
    # Step 1: For each time step, sample next state
    for t in range(1, n_steps):
        current_state = states[t-1]
        # Step 2: Transition probabilities from current state
        transition_probs = A[current_state, :]
        # Step 3: Sample next state
        states[t] = np.random.choice(K, p=transition_probs)
    
    return states

# Simulate a trajectory
n_steps = 500
states = simulate_markov_chain(A, initial_state=0, n_steps=n_steps, random_seed=42)

# Compute empirical state frequencies
empirical_freq = np.bincount(states, minlength=2) / n_steps

print("Simulation Results:")
print("="*50)
print(f"Simulated {n_steps} steps starting in Bull state\n")
print("Empirical frequencies:")
print(f"  Bull: {empirical_freq[0]:.4f}")
print(f"  Bear: {empirical_freq[1]:.4f}")
print("\nStationary distribution (theoretical):")
print(f"  Bull: {pi[0]:.4f}")
print(f"  Bear: {pi[1]:.4f}")

# Visualize trajectory
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: State trajectory
ax1 = axes[0]
ax1.fill_between(range(n_steps), states, alpha=0.6, step='mid')
ax1.set_ylabel('State', fontsize=12)
ax1.set_title('Simulated Market Regime Trajectory', fontsize=14)
ax1.set_yticks([0, 1])
ax1.set_yticklabels(['Bull', 'Bear'])
ax1.grid(True, alpha=0.3)

# Bottom: Cumulative state frequencies
ax2 = axes[1]
cumulative_bull = np.cumsum(states == 0) / (np.arange(n_steps) + 1)
cumulative_bear = np.cumsum(states == 1) / (np.arange(n_steps) + 1)
ax2.plot(cumulative_bull, label='Bull frequency', linewidth=2)
ax2.plot(cumulative_bear, label='Bear frequency', linewidth=2)
ax2.axhline(pi[0], color='green', linestyle='--', alpha=0.5, label=f'π(Bull) = {pi[0]:.3f}')
ax2.axhline(pi[1], color='red', linestyle='--', alpha=0.5, label=f'π(Bear) = {pi[1]:.3f}')
ax2.set_xlabel('Time Step', fontsize=12)
ax2.set_ylabel('Cumulative Frequency', fontsize=12)
ax2.set_title('Convergence to Stationary Distribution', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Exercise 1: Three-State Market Model

**Task:** Create a 3-state Markov chain representing Bull, Neutral, and Bear market regimes.

Design the transition matrix such that:
- Each regime has at least 70% self-persistence
- Transitions between Bull and Bear must go through Neutral
- Compute and visualize the stationary distribution

**Expected Output:** A 3x3 transition matrix and stationary distribution showing which regime is most common.

In [ ]:
# YOUR CODE HERE
# ---------------

# Step 1: Create the 3-state transition matrix
# States: 0 = Bull, 1 = Neutral, 2 = Bear
A_three_state = None  # Replace with your implementation

# Step 2: Verify it's a valid transition matrix
# (Check row sums equal 1 and no direct Bull->Bear or Bear->Bull transitions)

# Step 3: Compute stationary distribution
pi_three_state = None  # Replace with your implementation

# Step 4: Visualize as heatmap

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert A_three_state is not None, "Don't forget to create A_three_state!"
    assert A_three_state.shape == (3, 3), "Matrix should be 3x3"
    
    # Check row sums
    row_sums = A_three_state.sum(axis=1)
    assert np.allclose(row_sums, 1.0), "Each row must sum to 1"
    
    # Check self-persistence
    assert all(A_three_state[i, i] >= 0.7 for i in range(3)), \
        "Each state should have at least 70% self-persistence"
    
    # Check no direct Bull<->Bear transitions
    assert A_three_state[0, 2] == 0.0, "No direct Bull->Bear transition allowed"
    assert A_three_state[2, 0] == 0.0, "No direct Bear->Bull transition allowed"
    
    # Check stationary distribution
    assert pi_three_state is not None, "Don't forget to compute pi_three_state!"
    assert np.isclose(pi_three_state.sum(), 1.0), "Stationary distribution must sum to 1"
    assert np.allclose(pi_three_state, pi_three_state @ A_three_state), \
        "Stationary distribution should satisfy π = πA"
    
    print("✅ Exercise 1 passed!")
    print(f"\nYour stationary distribution: {pi_three_state}")
    print(f"Most common regime: {['Bull', 'Neutral', 'Bear'][np.argmax(pi_three_state)]}")

test_exercise_1()

<details>
<summary>Hint 1</summary>
Start by setting diagonal elements to 0.75 for high persistence. Then distribute the remaining 0.25 probability to allowed transitions.
</details>

<details>
<summary>Hint 2</summary>
For the Bull state (row 0), you can only transition to Bull or Neutral. For Bear state (row 2), you can only transition to Bear or Neutral.
</details>

## Exercise 2: Expected Hitting Time

**Task:** Compute the expected number of time steps to reach the Bear state starting from the Bull state.

The expected hitting time $h_i$ from state $i$ to a target state satisfies:
$$h_i = 1 + \sum_{j \neq \text{target}} a_{ij} h_j$$

This is a system of linear equations that can be solved.

**Expected Output:** A single number representing the average number of steps.

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_hitting_time(A: np.ndarray, start_state: int, target_state: int) -> float:
    """
    Compute expected hitting time from start_state to target_state.
    
    Parameters
    ----------
    A : ndarray (K, K)
        Transition matrix
    start_state : int
        Starting state
    target_state : int
        Target state to reach
    
    Returns
    -------
    hitting_time : float
        Expected number of steps
    """
    # YOUR IMPLEMENTATION HERE
    return None

# Compute expected time from Bull to Bear
expected_time = compute_hitting_time(A, start_state=0, target_state=1)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert expected_time is not None, "Don't forget to compute expected_time!"
    assert isinstance(expected_time, (int, float, np.number)), \
        "Expected time should be a number"
    assert expected_time > 0, "Expected time must be positive"
    
    # Check against simulation
    simulated_times = []
    for _ in range(1000):
        states = simulate_markov_chain(A, initial_state=0, n_steps=1000)
        bear_times = np.where(states == 1)[0]
        if len(bear_times) > 0:
            simulated_times.append(bear_times[0])
    
    sim_mean = np.mean(simulated_times)
    
    # Should be within 20% of simulated value
    assert abs(expected_time - sim_mean) / sim_mean < 0.2, \
        f"Your answer {expected_time:.2f} differs significantly from simulation {sim_mean:.2f}"
    
    print("✅ Exercise 2 passed!")
    print(f"\nExpected hitting time (Bull → Bear): {expected_time:.2f} steps")
    print(f"Simulated average: {sim_mean:.2f} steps")

test_exercise_2()

<details>
<summary>Hint 1</summary>
Set up the equation system (I - A_reduced) h = 1, where A_reduced excludes the target state row and column.
</details>

<details>
<summary>Hint 2</summary>
Use np.linalg.solve to solve the linear system. The solution gives hitting times from all non-target states.
</details>

## Exercise 3: Long-Run Regime Proportions

**Task:** Given daily market returns data, model regime persistence and predict long-run proportions.

1. Estimate transition probabilities from a simulated return series
2. Compute the stationary distribution
3. Compare with empirical frequencies

**Expected Output:** Estimated transition matrix and comparison plot.

In [ ]:
# Generate synthetic return data with regimes
np.random.seed(123)
true_states = simulate_markov_chain(A, initial_state=0, n_steps=1000)

# Generate returns based on states
# Bull: mean=0.1%, std=1%
# Bear: mean=-0.2%, std=2%
returns = np.where(
    true_states == 0,
    np.random.normal(0.001, 0.01, size=1000),
    np.random.normal(-0.002, 0.02, size=1000)
)

# YOUR CODE HERE
# ---------------

def estimate_transition_matrix(states: np.ndarray, n_states: int) -> np.ndarray:
    """
    Estimate transition matrix from observed state sequence.
    
    Parameters
    ----------
    states : ndarray (T,)
        Observed state sequence
    n_states : int
        Number of states
    
    Returns
    -------
    A_est : ndarray (n_states, n_states)
        Estimated transition matrix
    """
    # YOUR IMPLEMENTATION HERE
    return None

# You would first need to classify returns into Bull/Bear states
# Then estimate the transition matrix
A_estimated = None  # Replace with your implementation
pi_estimated = None  # Replace with your implementation

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert A_estimated is not None, "Don't forget to estimate the transition matrix!"
    assert A_estimated.shape == (2, 2), "Matrix should be 2x2"
    
    # Check validity
    assert np.all(A_estimated >= 0), "All probabilities must be non-negative"
    assert np.allclose(A_estimated.sum(axis=1), 1.0), "Rows must sum to 1"
    
    # Check it's close to true matrix
    error = np.max(np.abs(A_estimated - A))
    assert error < 0.15, f"Estimated matrix differs too much from true matrix (error={error:.3f})"
    
    assert pi_estimated is not None, "Don't forget to compute stationary distribution!"
    
    print("✅ Exercise 3 passed!")
    print(f"\nTrue transition matrix:\n{A}")
    print(f"\nEstimated transition matrix:\n{A_estimated}")
    print(f"\nMax error: {error:.4f}")

test_exercise_3()

<details>
<summary>Hint 1</summary>
First classify returns into Bull (positive) and Bear (negative) states using return > 0 as the threshold.
</details>

<details>
<summary>Hint 2</summary>
Count transitions: how many times did you go from Bull to Bull, Bull to Bear, etc.? Then normalize each row.
</details>

## Summary

### Key Takeaways

1. **Markov property**: Future depends only on present, not past history
2. **Transition matrices**: Encode state dynamics with row-stochastic probabilities
3. **Multi-step transitions**: Computed via matrix powers $A^n$
4. **Stationary distribution**: Long-run state probabilities, independent of start
5. **Simulation**: Generate trajectories by sampling from transition probabilities

### What's Next

In the next module, we'll extend to **Hidden Markov Models** where:
- States are hidden (unobserved)
- We observe emissions that depend on states
- We need algorithms to infer states from observations

### Additional Resources

- Norris, J.R. (1997). *Markov Chains*. Cambridge University Press.
- Rabiner, L.R. (1989). "A Tutorial on Hidden Markov Models". Proceedings of IEEE.
- Hamilton, J.D. (1989). "A New Approach to the Economic Analysis of Nonstationary Time Series". Econometrica.

---

**Next Notebook:** `02_environment_setup.ipynb` - Setting up Python libraries for HMM work